In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_cfb_2024_passing():
    url = "https://www.sports-reference.com/cfb/years/2024-passing.html"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Locate table with ID 'passing_standard'
    table = soup.find("table", id="passing_standard")
    if not table:
        raise ValueError("Could not find table with id='passing_standard'.")

    # Extract column headers from the first data row
    tbody = table.find("tbody")
    first_valid_row = next(row for row in tbody.find_all("tr") if row.find_all("td"))
    columns = [td['data-stat'] for td in first_valid_row.find_all("td")]

    # Extract data rows
    data = []
    for row in tbody.find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue  # skip sub-header rows
        tds = row.find_all("td")
        if not tds:
            continue
        values = [td.get_text(strip=True) for td in tds]
        data.append(values)

    # Create DataFrame
    df = pd.DataFrame(data, columns=columns)

    # Convert number-looking strings
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

    return df

if __name__ == "__main__":
    df = scrape_cfb_2024_passing()
    print(df.head())
    df.to_csv("cfb_2024_passing_standard.csv", index=False)
